In [0]:

%run ./00_config

### Step 1: Silver Layer Data Cleansing, Quality Checks & Deduplication
* **Source:** Bronze Delta table (`youtube_videos_bronze`).
* **Deduplication:** Window function partitioned by `video_id`, ordered by trending date descending to keep only the latest state.
* **Cleansing:** String trimming, handling null descriptions, and casting numerical attributes.
* **Data Quality Guardrails:** Filtering out corrupted records with negative view counts.
* **Audit Metadata:** Appending `_silver_processed_at` timestamp.

In [0]:
from pyspark.sql.functions import col, trim, coalesce, lit, current_timestamp, row_number
from pyspark.sql.window import Window

In [0]:
df_bronze_source = spark.table(source_bronze_table)

df_clefned = (
    df_bronze_source
    .filter((col("views") >= 0) & (col("likes") >= 0))
    .withColumn("title", trim(col("title")))
    .withColumn("channel_title", trim(col("channel_title")))
    .withColumn("description", coalesce(trim(col("description")), lit("No description")))
    .withColumn("_silver_processed_at", current_timestamp())                                  
)

windows_spec = Window.partitionBy(business_key).orderBy(col("trending_date").desc(), col("_ingestion_timestamp").desc())

df_silver_prepared = (
    df_clefned
    .withColumn("row_num", row_number().over(windows_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

display(df_silver_prepared.limit(5))

### Step 2: MERGE INTO Pipeline (SCD Type 1)
* **Objective:** Implement Slowly Changing Dimensions Type 1 using Delta `MERGE`.
* **Logic:**
  * **MATCHED:** Update existing video metrics (views, likes) and `_silver_processed_at` timestamp.
  * **NOT MATCHED:** Insert completely new trending videos.
* **Initialization:** If the target Silver table does not exist, it is created automatically before merging.

In [0]:
from delta.tables import DeltaTable

if not spark.catalog.tableExists(target_silver_table):
    (
        df_silver_prepared.write
        .format("delta")
        .saveAsTable(target_silver_table)
    )

else:
    target_table = DeltaTable.forName(spark, target_silver_table)
    (
        target_table.alias("target")
        .merge(
            source = df_silver_prepared.alias("source"),
            condition = f"target.{business_key} = source.{business_key}"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from delta.tables import DeltaTable

df_existing = (
    spark.table(target_silver_table)
    .withColumn("is_current", lit(True))
    .withColumn("valid_from", col("_silver_processed_at"))
    .withColumn("valid_to", lit(None).cast("timestamp"))
)

(
    df_existing.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_silver_table)
)

sample_id = "-0CMnp02rNY"

df_incoming_update = (
    spark.table(target_silver_table)
    .filter(col("video_id") == sample_id)
    .withColumn("views", col("views") + 50000)
)

target_delta_table = DeltaTable.forName(spark, target_silver_table)

(
    target_delta_table.alias("target")
    .merge(
        source = df_incoming_update.alias("source"),
        condition = f"target.{business_key} = source.{business_key}"
    )
    .whenMatchedUpdate(
        condition = "target.views <> source.views",
        set = {
            "is_current": lit(False),
            "valid_to": current_timestamp()
        }
    )
    .execute()
)

df_new_active_record = (
    df_incoming_update
    .withColumn("is_current", lit(True))
    .withColumn("valid_from", current_timestamp())
    .withColumn("valid_to", lit(None).cast("timestamp"))
)

(
    df_new_active_record.write
    .format("delta")
    .mode("append")
    .saveAsTable(target_silver_table)
)

display(
     spark.table(target_silver_table)
     .filter(col("video_id") == sample_id)
     .select("video_id", "views", "is_current", "valid_from", "valid_to")
     .orderBy(col("valid_from").desc())
)

### Step 4: Schema Enforcement, Evolution & Data Contracts

**Schema Enforcement:** Delta Lake strictly validates incoming DataFrames. If a rogue column appears, the write is aborted, preventing data swamp scenarios.

In [0]:
# from pyspark.sql.functions import lit

# df_rogue = (
#     spark.table(target_silver_table)
#     .limit(1)
#     .withColumn("unexpected_column", lit("unexpected"))
# )

# (
#     df_rogue.write
#     .format("delta")
#     .mode("append")
#     .saveAsTable(target_silver_table)
# )


**Schema Evolution:** By enabling `.option("mergeSchema", "true")`, we explicitly instruct Delta to append the new column to the table metadata.

In [0]:
df_rogue = (
    spark.table(target_silver_table)
    .limit(1)
    .withColumn("unexpected_column", lit("unexpected"))
)

(
    df_rogue.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(target_silver_table)
)

* **Column Mapping:** Setting `delta.columnMapping.mode = 'name'` allows metadata-only column renames and drops without requiring expensive parquet file rewrites.

In [0]:

spark.sql(f"ALTER TABLE {target_silver_table} SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name')")

df_renamed = (
    spark.table(target_silver_table)
    .withColumnRenamed("unexpected_column", "verified_column")
)

(
    df_renamed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_silver_table)
)

#### Data Contracts vs. Silent Evolution
While `autoMerge` (Schema Evolution) is convenient, it can be dangerous in production (silent evolution). If an upstream team changes a column type from `INT` to `STRING` without notice, downstream BI dashboards will crash. 
**Data Contracts** are the solution: they act as a formal, version-controlled agreement between data producers and consumers (often enforced via tools like Unity Catalog constraints or JSON schemas). With data contracts, any breaking schema change is blocked at the source/producer level *before* it ever reaches the pipeline.